# 06 · MCP – plug J.A.R.V.I.S. into any toolbox

The **Model Context Protocol (MCP)** is an open standard for exposing tools to AI. An MCP server lists tools (name, description, schema); an MCP client (here: Foundry Agent Service) lets the model call them.

| ⚙️ Setting on the `mcp` tool | 🎯 Purpose |
|---|---|
| `server_label` | unique name of the server inside the agent |
| `server_url` | remote MCP endpoint (must be reachable from Foundry) |
| `allowed_tools` | allow-list of tool names (default: all) |
| `require_approval` | `always` (default) \| `never` \| per-tool lists |
| `project_connection_id` | project connection holding credentials |

Deep dive: [docs/mcp-a2a-skills.md](../docs/mcp-a2a-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

from azure.ai.projects.models import PromptAgentDefinition, MCPTool
oai = openai_client()
created_agents = []

## 1 · A public MCP server: Microsoft Learn
No credentials needed. `require_approval="always"` means **you** see every tool call before it runs.

In [ ]:
learn = MCPTool(server_label="learn", server_url="https://learn.microsoft.com/api/mcp", require_approval="always")
agent = project().agents.create_version(
    agent_name="jarvis-nb06-learn",
    definition=PromptAgentDefinition(model=model("chat"), instructions="Use the MCP tools to answer. Cite the page titles.", tools=[learn]),
)
created_agents.append(agent.name)

# approve only calls to the server we expect
resp = ask_agent(agent, "How do I create a conversation with the Foundry Agent Service in Python? Two sentences.",
                 approve=lambda item: item.server_label == "learn")
print(resp.output_text)
print(tool_calls(resp))

### Restrict the tools
A server may expose dozens of tools. `allowed_tools` is an allow-list: the model never even sees the others. Use it and `require_approval` for anything that writes data.

In [ ]:
learn_ro = MCPTool(server_label="learn", server_url="https://learn.microsoft.com/api/mcp",
                   allowed_tools=["microsoft_docs_search"], require_approval="never")
agent2 = project().agents.create_version(
    agent_name="jarvis-nb06-learn",
    definition=PromptAgentDefinition(model=model("chat"), instructions="Use the MCP tools to answer. One sentence.", tools=[learn_ro]),
)
resp = ask_agent(agent2, "What is a Foundry project endpoint?")
print(resp.output_text, tool_calls(resp))

## 2 · Our own MCP server: `stark-mcp`
[src/stark-mcp](../src/stark-mcp/function_app.py) is an **Azure Function** with the Functions MCP extension: four tools (`list_suits`, `get_suit`, `get_workshop_status`, `check_invoice_approval`). `azd up` deployed it, and the postdeploy hook created the project connection `stark-mcp` that holds the function key – **the key never appears in the agent definition**.

Check that the pieces exist:

In [ ]:
mcp_url = cfg("STARK_MCP_URL")
print("MCP endpoint:", mcp_url)
try:
    conn = project().connections.get("stark-mcp")
    print("connection  :", conn.name, "-", getattr(conn, "type", ""))
except Exception as exc:
    print("connection 'stark-mcp' missing:", type(exc).__name__, "-> run: python scripts/connect_mcp.py")

Deploy the agent from YAML ([agents/stark-ops.yaml](../agents/stark-ops.yaml); chapter 08 explains the script) and ask it about suits.

In [ ]:
sh("python3 scripts/deploy_agents.py agents/stark-ops.yaml")
for q in ["Which suits are still operational?", "What is the status of the fabrication bay?"]:
    r = ask_agent("stark-ops", q)
    print(f"> {q}\n{r.output_text}  {tool_calls(r)}\n")

## 3 · Trust: treat tool output as untrusted
Descriptions and results of a remote MCP server flow straight into the model's context. A hostile or compromised server can embed instructions ("ignore previous instructions...") – **indirect prompt injection**. Defences used here: allow-lists, approvals for risky tools, least-privilege connections. Chapter 14 attacks this on purpose.

## Clean up

In [ ]:
for name in created_agents + ["stark-ops"]:
    delete_agent(name)

## Recap
MCP gives agents a **standard plug**: one server, many agents, any client. Remote only – a local `stdio` server must be hosted (Functions, Container Apps) first.

> 💡 Next: [07 · Skills](07_agent_skills.ipynb) 🧪 – reusable know-how.